<a href="https://colab.research.google.com/github/jyothsnareddym/BIS_LAB/blob/main/1BF24CS180_W3_AC0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

def calculate_total_completion_time(schedule, processing_times):
    """Calculates the sum of completion times for all jobs in a given schedule."""
    current_clock = 0
    total_completion_time = 0

    for job in schedule:
        current_clock += processing_times[job]
        total_completion_time += current_clock

    return total_completion_time


def roulette_wheel_select(scores):
    """Picks a job randomly, weighted by its desirability score."""
    total_score = sum(scores.values())
    pick = random.uniform(0, total_score)
    current = 0

    for job, score in scores.items():
        current += score
        if current >= pick:
            return job

    return list(scores.keys())[-1]


def aco_job_scheduling(jobs, processing_times, num_ants=10, max_iterations=50,
                       alpha=1.0, beta=2.0, rho=0.1, Q=100.0):
    """Finds the optimal job execution order to minimize total waiting/completion time."""

    all_nodes = ['START'] + jobs
    pheromone = {i: {j: 1.0 for j in jobs} for i in all_nodes}

    heuristic = {j: 1.0 / processing_times[j] for j in jobs}

    best_schedule = None
    best_completion_time = float('inf')

    for iteration in range(max_iterations):
        all_ant_schedules = []

        # --- Step A: Construct Solutions ---
        for ant in range(num_ants):
            current_schedule = []
            unscheduled_jobs = list(jobs)
            current_job = 'START'

            while unscheduled_jobs:
                probability_scores = {}

                for next_job in unscheduled_jobs:
                    tau = pheromone[current_job][next_job] ** alpha
                    eta = heuristic[next_job] ** beta
                    probability_scores[next_job] = tau * eta


                selected_job = roulette_wheel_select(probability_scores)

                current_schedule.append(selected_job)
                unscheduled_jobs.remove(selected_job)
                current_job = selected_job

            total_time = calculate_total_completion_time(current_schedule, processing_times)
            all_ant_schedules.append((current_schedule, total_time))

            if total_time < best_completion_time:
                best_completion_time = total_time
                best_schedule = current_schedule

        for i in all_nodes:
            for j in jobs:
                pheromone[i][j] *= (1.0 - rho)

        for schedule, total_time in all_ant_schedules:
            deposit_amount = Q / total_time

            pheromone['START'][schedule[0]] += deposit_amount

            for k in range(len(schedule) - 1):
                job_a = schedule[k]
                job_b = schedule[k + 1]
                pheromone[job_a][job_b] += deposit_amount


    return best_schedule, best_completion_time


if __name__ == "__main__":
    job_list = ['Job_1', 'Job_2', 'Job_3', 'Job_4', 'Job_5']
    job_durations = {
        'Job_1': 10,
        'Job_2': 2,
        'Job_3': 15,
        'Job_4': 5,
        'Job_5': 8
    }

    best_order, min_wait_time = aco_job_scheduling(
        jobs=job_list,
        processing_times=job_durations,
        num_ants=10,
        max_iterations=50,
        alpha=1.0,
        beta=2.0,
        rho=0.1,
        Q=100.0
    )

    print("Optimal Job Schedule:", " -> ".join(best_order))
    print("Minimum Total Completion Time:", min_wait_time, "minutes")


Optimal Job Schedule: Job_2 -> Job_4 -> Job_5 -> Job_1 -> Job_3
Minimum Total Completion Time: 89 minutes
